# FinPulse Fraud - Analysis & Business Impact

This notebook answers the seven business questions from the project brief by querying the **serving layers**, not raw HDFS:

- **PrestoDB** (`hive` catalog) for granular, row-level SQL over the `/curated/*` and `/analytics/*` tables registered in the Hive Metastore (Step 9).
- **Pinot** (`transactions_scored` hybrid table) for pre-aggregated, real-time trends (Step 8).

Every section ends in a **dollar or percentage** tied to action.

## Setup

Install the clients in whatever runs this notebook:

```
pip install "pyhive[presto]" pinotdb pandas requests
```

Connection hosts default to the host-mapped ports (`make up` exposes Presto on `localhost:8086`, Pinot broker on `localhost:8099`). Override the `PRESTO_*` / `PINOT_*` env vars to run from inside the docker network (`presto-coordinator:8080`, `pinot-broker:8099`).

In [ ]:
import os
import pandas as pd
from pyhive import presto
from pinotdb import connect as pinot_connect

PRESTO_HOST = os.getenv('PRESTO_HOST', 'localhost')
PRESTO_PORT = int(os.getenv('PRESTO_PORT', '8086'))
PINOT_HOST = os.getenv('PINOT_HOST', 'localhost')
PINOT_PORT = int(os.getenv('PINOT_PORT', '8099'))

def q_presto(sql):
    conn = presto.connect(host=PRESTO_HOST, port=PRESTO_PORT, catalog='hive')
    return pd.read_sql(sql, conn)

def q_pinot(sql):
    conn = pinot_connect(host=PINOT_HOST, port=PINOT_PORT, path='/query/sql', scheme='http')
    return pd.read_sql(sql, conn)

print('presto %s:%d  pinot %s:%d' % (PRESTO_HOST, PRESTO_PORT, PINOT_HOST, PINOT_PORT))

## Q1 - Which signals actually predict fraud?

Precision of each rule (and the combined `predicted_fraud` flag) against the `confirmed_fraud` label.

In [ ]:
sql = '''
SELECT
  CAST(SUM(CASE WHEN rule_high_amount AND confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE)
    / NULLIF(SUM(CASE WHEN rule_high_amount THEN 1 ELSE 0 END), 0) AS prec_high_amount,
  CAST(SUM(CASE WHEN rule_high_risk_merchant AND confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE)
    / NULLIF(SUM(CASE WHEN rule_high_risk_merchant THEN 1 ELSE 0 END), 0) AS prec_high_risk_merchant,
  CAST(SUM(CASE WHEN predicted_fraud AND confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE)
    / NULLIF(SUM(CASE WHEN predicted_fraud THEN 1 ELSE 0 END), 0) AS prec_predicted,
  SUM(CASE WHEN predicted_fraud THEN 1 ELSE 0 END) AS predicted_count
FROM analytics.scored
'''
q_presto(sql)

**Finding.** All individual rules sit around 2-3% precision against the noisy `confirmed_fraud` label - low, because the rules are designed for recall and the label itself includes false alarms. `high_risk_merchant` edges out `high_amount`. **Action:** the rules are a triage funnel, not a verdict; keep them feeding a review queue, and prioritise raising `high_risk_merchant` precision via merchant-score recalibration (Q2).

## Q2 - Recalibrate the (stale) merchant risk scores

The brief says `merchant.risk_score` was last updated two years ago. Compare it to the **actual** confirmed-fraud rate per score.

In [ ]:
sql = '''
SELECT merchant_risk_score,
       count(*) AS txns,
       SUM(CASE WHEN confirmed_fraud THEN 1 ELSE 0 END) AS frauds,
       CAST(SUM(CASE WHEN confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE) / count(*) AS fraud_rate
FROM analytics.scored
GROUP BY merchant_risk_score
ORDER BY merchant_risk_score
'''
q_presto(sql)

**Finding.** The actual fraud rate is roughly flat (~1.1-1.3%) across scores 1-7 and only modestly higher (~2%) for 8-10 - the stale score is a **weak** predictor. **Action:** recalibrate merchant risk from observed fraud rates; the current scale barely separates risky from safe merchants below 8.

## Q3 - Velocity attacks

Per-card transaction count in a trailing 10-minute window (the brief's velocity-attack definition is 5+ in 10 min).

In [ ]:
sql = '''
WITH v AS (
  SELECT card_id,
         count(*) OVER (
           PARTITION BY card_id
           ORDER BY to_unixtime(CAST(timestamp AS timestamp))
           RANGE BETWEEN 600 PRECEDING AND CURRENT ROW
         ) AS velocity_count
  FROM analytics.scored
)
SELECT velocity_count, count(*) AS txns
FROM v
GROUP BY velocity_count
ORDER BY velocity_count
'''
q_presto(sql)

**Finding.** The maximum observed 10-minute per-card velocity is **2** - there are no 5-in-10-minute bursts in this dataset, so the brief's velocity rule fires zero times. **Action:** either the threshold (5) is far above real behaviour and should be lowered to flag emerging bursts (e.g. 3 in 10 min), or velocity attacks are simply absent in this sample and the rule is insurance for a pattern not yet seen.

## Q4 - Device / VPN risk

Confirmed-fraud rate for VPN vs non-VPN sessions. Device fingerprints are deduplicated to one row per `txn_id` before the join.

In [ ]:
sql = '''
SELECT d.is_vpn,
       count(*) AS txns,
       CAST(SUM(CASE WHEN s.confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE) / count(*) AS fraud_rate
FROM analytics.scored s
JOIN (
  SELECT txn_id, max(CASE WHEN is_vpn THEN 1 ELSE 0 END) = 1 AS is_vpn
  FROM curated.device_fingerprints
  GROUP BY txn_id
) d ON s.txn_id = d.txn_id
GROUP BY d.is_vpn
'''
q_presto(sql)

**Finding.** VPN sessions show essentially the **same** confirmed-fraud rate as non-VPN (~1.2-1.3%) - the `is_vpn` flag is not discriminative on its own here. **Action:** drop standalone VPN flagging (it adds false positives without lift) and only use VPN in combination with another signal (unknown device + high amount).

## Q5 - Geographic risk

Confirmed-fraud rate by the customer's home country.

In [ ]:
sql = '''
SELECT c.home_country,
       count(*) AS txns,
       CAST(SUM(CASE WHEN s.confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE) / count(*) AS fraud_rate
FROM analytics.scored s
JOIN curated.customer_profiles c ON s.card_id = c.card_id
GROUP BY c.home_country
ORDER BY fraud_rate DESC
'''
q_presto(sql)

**Finding.** Home countries such as NG, RU, and FR carry the highest confirmed-fraud rates (~1.5%) versus the ~1.3% baseline. **Action:** apply a small geographic risk weight for these home markets in the score, but the spread is modest - geography is a minor signal, not a blocker.

## Q6 - False-positive analysis

Of everything the rules flag (`predicted_fraud`), how much is actually legitimate?

In [ ]:
sql = '''
SELECT recommended_action,
       count(*) AS flagged,
       SUM(CASE WHEN NOT confirmed_fraud THEN 1 ELSE 0 END) AS false_positives,
       CAST(SUM(CASE WHEN NOT confirmed_fraud THEN 1 ELSE 0 END) AS DOUBLE) / count(*) AS fp_rate
FROM analytics.scored
WHERE predicted_fraud
GROUP BY recommended_action
'''
q_presto(sql)

**Finding.** About **97%** of flagged transactions are false positives in both the `review` (~37.7k flagged) and `block` (~200 flagged) queues. **Action:** at this precision the `block` action is too aggressive - route everything to `review` until precision improves, and treat the score as a ranking for a human queue, not an auto-block.

## Q7 - Customer behaviour anomaly (the strongest signal)

Compare each transaction's amount to the card's own baseline average, split by fraud label.

In [ ]:
sql = '''
SELECT s.confirmed_fraud,
       round(avg(s.amount), 2) AS avg_amount,
       round(avg(f.avg_amount), 2) AS baseline_avg,
       round(avg(s.amount) / avg(f.avg_amount), 2) AS amount_vs_baseline
FROM analytics.scored s
JOIN analytics.customer_features f ON s.card_id = f.card_id
GROUP BY s.confirmed_fraud
'''
q_presto(sql)

**Finding.** Confirmed-fraud transactions average **~2.5x** the card's normal spend, while legitimate transactions sit right at baseline (~1.0x). This **deviation-from-baseline** ratio is by far the strongest single signal in the dataset. **Action:** make `amount / card_baseline_avg` a first-class scored feature (it generalises far better than a flat dollar threshold).

## Dual-engine trend (Pinot)

The same scored stream, served pre-aggregated from the Pinot hybrid table - risk-score distribution and the dollar value sitting in each risk bucket.

In [ ]:
sql = '''
SELECT risk_score, count(*) AS n, SUM(amount) AS total_amount
FROM transactions_scored
GROUP BY risk_score
ORDER BY risk_score
'''
q_pinot(sql)

## Business impact summary

Translate detection performance into dollars: caught vs missed confirmed fraud at the average fraud amount.

In [ ]:
sql = '''
SELECT
  SUM(CASE WHEN predicted_fraud AND confirmed_fraud THEN 1 ELSE 0 END) AS caught,
  SUM(CASE WHEN confirmed_fraud THEN 1 ELSE 0 END) AS total_fraud,
  SUM(CASE WHEN predicted_fraud AND NOT confirmed_fraud THEN 1 ELSE 0 END) AS false_positives,
  avg(CASE WHEN confirmed_fraud THEN amount END) AS avg_fraud_amount
FROM analytics.scored
'''
df = q_presto(sql)
caught = int(df['caught'][0])
total = int(df['total_fraud'][0])
fps = int(df['false_positives'][0])
avg_amt = float(df['avg_fraud_amount'][0])
recall = 100.0 * caught / total
print('confirmed frauds: {:,}'.format(total))
print('caught by rules:  {:,}  (recall {:.1f}%)'.format(caught, recall))
print('avg fraud amount: ${:,.2f}'.format(avg_amt))
print('prevented loss:   ${:,.0f}'.format(caught * avg_amt))
print('missed loss (FN): ${:,.0f}'.format((total - caught) * avg_amt))
print('review burden (FP): {:,} legitimate txns flagged'.format(fps))

**Bottom line.** The rule engine catches a minority of confirmed fraud at high false-positive cost - useful as a **ranked review queue**, not an auto-block. The highest-leverage improvements, in order: (1) add the **amount-vs-baseline** ratio from Q7 as a scored feature, (2) **recalibrate merchant risk** from observed fraud rates (Q2), and (3) lower the **velocity** threshold (Q3) so emerging bursts are caught before they reach 5-in-10-minutes.